In [42]:
import pandas as pd

In [43]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'tip_amount', 'total_amount']
df = pd.read_parquet(url, columns=columns)
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [44]:
df['lpep_pickup_datetime'] = df['lpep_pickup_datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
df['lpep_dropoff_datetime'] = df['lpep_dropoff_datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')

In [45]:
df['passenger_count'] = df['passenger_count'].astype('Int32')

In [ ]:
df_sorted = df.sort_values(by='lpep_pickup_datetime')

lpep_pickup_datetime         str
lpep_dropoff_datetime        str
PULocationID               int32
DOLocationID               int32
passenger_count            Int32
trip_distance            float64
tip_amount               float64
total_amount             float64
dtype: object

In [ ]:
from time import time
import json
from kafka import KafkaProducer
import numpy as np


topic_name = 'green-trips'

def json_serializer(data):
    def default_handler(x):
        if pd.isna(x):
            return None
        if isinstance(x, np.generic):
            return x.item()
        return x
    return json.dumps(data, default=default_handler).encode('utf-8')


server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

t0 = time()
row_count = 0
for row in df_sorted.itertuples(index=False):
    row_count += 1
    producer.send(topic_name, value=row._asdict())

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')
print(f'sent {row_count} records')

took 4.08 seconds
sent 49416 records
